In [8]:
import os
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time

# ============ 配置 ============
client = OpenAI(
    base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
    api_key='ida_TbtBzRyj8HeV7kmxkeRsgImHnYelRIUMQ3vW9VIf'
)

MODEL = 'qwen-2.5-72b-instruct'

# ============ Few-Shot Prompt ============
FEW_SHOT_PROMPT_TEMPLATE = """You are a professional search system optimization expert. Your task is to determine if Pseudo-Relevance Feedback (PRF) should be applied to expand a query.

Here are some reference examples from user studies:

### Example 1
Query: "definition of a sigmet"
Retrieved Documents:
[1] [Score: 28.22] Definition of SIGMET: a notice of significant hazardous weather conditions (such as extreme turbulence, icing, or poor visibility) provided to aircraft pilots before takeoff.
[2] [Score: 25.97] SIGMET, or Significant Meteorological Information, is a weather advisory concerning the safety of all aircraft. There are two types: convective and non-convective.
[3] [Score: 25.51] SIGMET is a weather advisory for aircraft safety. Two types exist - convective and non-convective. Non-convective SIGMETs are issued for severe turbulence, severe icing, or instrument meteorological conditions.
[4] [Score: 25.15] SIGMET weather advisory with detailed criteria: severe turbulence over 3,000-square-mile area, severe icing over similar area, or IMC due to dust, sand, or volcanic ash.
[5] [Score: 24.22] Medical Definition of Sigmoid: the lower colon in human anatomy, shaped like the Greek letter sigma (C-shaped).

Analysis: The query seeks a technical aviation term definition. Top-4 documents are highly relevant and authoritative, all consistently defining SIGMET as a meteorological safety advisory for aircraft. These documents share common domain terminology (meteorological, aircraft, weather, safety, turbulence, icing). Doc-5 is a false match (medical term "sigmoid" vs aviation term "sigmet") but has low relevance score. The query intent is clear and unambiguous. Applying PRF can extract shared aviation/meteorological terminology (hazardous weather, aircraft safety, advisory, turbulence, visibility) to find more comprehensive explanations of aviation weather systems and related safety protocols.
Decision: YES

### Example 2
Query: "who is robert gray"
Retrieved Documents:
[1] [Score: 25.22] Robert Gray (1755-1806): American sea captain, first U.S. ship to circumnavigate the globe, explorer of Columbia River.
[2] [Score: 23.80] Captain Robert Gray discovered a great river and named it Columbia after his ship. His discovery formed basis of America's claim to Oregon Territory.
[3] [Score: 23.71] Robert Gray: Democratic candidate for Mississippi governor in 2015 elections, truck driver and former firefighter. Won primary but lost to incumbent Phil Bryant.
[4] [Score: 23.17] John Gray (born 1951): American relationship counselor, lecturer and author, associated with Maharishi Mahesh Yogi.
[5] [Score: 23.09] George Gray (born 1967): American game show host, announcer, and comedian from Missouri.

Analysis: Query about a person with multiple namesakes. Top-2 documents focus on the historical explorer Robert Gray (18th century maritime figure), while Doc-3 mentions a modern Mississippi politician with the same name. Docs-4,5 are false matches (different first names). Despite some ambiguity, the historical figure dominates the top results with high relevance. PRF can extract biographical and contextual terms (captain, explorer, Columbia River, maritime, circumnavigate, Oregon Territory) to strengthen retrieval of related historical exploration content and disambiguate from modern namesakes.
Decision: YES

### Example 3
Query: "what is theraderm used for"
Retrieved Documents:
[1] [Score: 23.62] Theraderm cream's main ingredient is lanolin from sheep's wool, used in skincare for boosting natural moisture barrier and drawing moisture to application site.
[2] [Score: 22.96] Theraderm is a manufacturer of clinical-grade skincare products: anti-aging creams, moisturizers, and acne system (Reversion brand). Founded 1996 by Dr. James Beckman.
[3] [Score: 21.86] A thermopile is a thermoelectric device consisting of thermocouples connected in series, widely used in non-contact temperature measurement and monitoring systems.
[4] [Score: 21.64] A theraband is a latex resistance band or tube used for physical therapy and light strength training, commonly used by athletes and dancers to strengthen feet.
[5] [Score: 21.59] A thermometer is a device used to measure temperature, used in healthcare to measure and monitor body temperature.

Analysis: The query seeks information about Theraderm skincare products. However, Top-5 documents show catastrophic topic drift. Only Docs-1,2 are relevant (skincare), while Docs-3,4,5 are completely unrelated - matching on similar spelling prefixes (thermo-/thera-) but covering physics and fitness domains. The result set is highly fragmented. Applying PRF would catastrophically mix terminology from these disparate domains (lanolin, thermocouples, resistance training, temperature), creating a nonsensical expanded query and causing massive query drift.
Decision: NO

### Example 4
Query: "causes of military suicide"
Retrieved Documents:
[1] [Score: 25.34] Canadian military study over 25 years: suicide was third leading cause of military deaths after motor vehicle accidents and cancer. Questions effectiveness of suicide prevention programs.
[2] [Score: 24.44] Suicide surpassed war as military's leading cause of death. War was the leading cause 2004-2011, but suicides became top cause of death for troops in 2012 and 2013.
[3] [Score: 24.29] Similar reporting: suicide surpassed war as leading cause of military death in 2012-2013 according to Pentagon medical statistical analysis.
[4] [Score: 24.21] WHO World Health Report 2004: deaths from intentional injuries (war, violence, suicide) were 2.8% of all deaths; unintentional injury was 6.2%.
[5] [Score: 24.14] Tragic anecdote about soldier Michael who saluted and shot himself. States suicide is now leading cause of death amongst active-duty soldiers.

Analysis: The query asks for "causes" (reasons/risk factors), but the retrieved documents primarily report "statistics" and "rankings" (leading cause of death, rates). The results are factually relevant to the topic but fail to address the specific "why" aspect of the user intent. They lack discussion of psychological factors, PTSD, or deployment stress. Applying PRF would likely extract statistical terms (leading cause, death rates, 2012-2013, fatalities) rather than explanatory causal terms. This would reinforce the focus on statistics rather than pivoting to the underlying causes the user requested.
Decision: NO

### Example 5
Query: "what is famvir prescribed for"
Retrieved Documents:
[1] [Score: 24.36] Famciclovir (Famvir) is a drug used for the treatment of genital herpes, cold sores, shingles, and chickenpox. Side effects, warnings, and precautions should be discussed.
[2] [Score: 24.14] Famciclovir (Famvir) is a prescription antiviral medication used to treat: Herpes simplex infections (cold sores and genital herpes) and Shingles.
[3] [Score: 23.94] Famciclovir (Famvir) is sometimes used to treat the herpes virus that causes cold sores and genital herpes, as well as the virus that causes shingles.
[4] [Score: 23.69] Famvir is approved by the FDA for the treatment of cold sores. Taking a single high dose can shorten the herpes infection.
[5] [Score: 23.48] Famciclovir (Famvir) is a prescription antiviral used to treat: Shingles in people with normal immune systems and recurring genital herpes.

Analysis: The query asks for the specific medical indications of Famvir. The top retrieved documents are excellent: they are highly consistent, authoritative, and directly list the exact conditions (genital herpes, cold sores, shingles, chickenpox). The user's information need is fully satisfied by the top results. Applying PRF in this case provides little marginal benefit and carries a risk of introducing unnecessary noise (e.g., specific side effects, dosage details, or generic antiviral terms) that could dilute the high precision of the current ranking. Since the answer is already saturated and precise, no expansion is needed.
Decision: NO

---
Now please analyze the following query:

Query: "{query}"
Retrieved Documents:
{documents}

Please provide your analysis following the same format:
Analysis: <Your detailed reasoning considering: 1) query intent clarity 2) initial retrieval quality 3) topic consistency 4) whether PRF will help or harm>
Decision: <YES or NO>
"""

# ============ 加载数据 ============
print("加载数据...")
df_preference = pd.read_csv('preference.csv')
df_queries = pd.read_csv('result_with_text.csv')
df_colbert = pd.read_csv('df_colbert_deduped.csv')

if 'query_text' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query_text'].first().to_dict()
elif 'query' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query'].first().to_dict()

user_study_qids = df_preference['qid'].tolist()[:10]
print(f"测试 User Study 的前 10 个查询")

# ============ 生成并打印完整输出 ============
results = []

for i, qid in enumerate(user_study_qids, 1):
    print("\n" + "="*80)
    print(f"查询 {i}/10 - QID: {qid}")
    print("="*80)
    
    query = qid_to_query.get(qid, f"[Query not found for QID {qid}]")
    print(f"Query: {query}")
    
    # 获取真实标签
    ground_truth_row = df_preference[df_preference['qid'] == qid]
    if len(ground_truth_row) > 0:
        ground_truth_preference = ground_truth_row['preference'].iloc[0]
        ground_truth_ratio = ground_truth_row['b_preference_ratio'].iloc[0]
        print(f"Ground Truth: {ground_truth_preference} (ratio: {ground_truth_ratio:.3f})")
    else:
        ground_truth_preference = "Unknown"
        ground_truth_ratio = None
        print(f"Ground Truth: Unknown")
    
    # 获取 Top-5
    top5 = df_colbert[df_colbert['qid'] == qid].nlargest(5, 'score')
    
    if len(top5) == 0:
        print(f"⚠️ 没有检索结果")
        continue
    
    print(f"\nTop-5 文档:")
    for idx, (_, row) in enumerate(top5.iterrows(), 1):
        print(f"  [{idx}] Score: {row['score']:.2f}")
        print(f"      {str(row['passage_text'])[:150]}...")
    
    # 格式化文档
    docs_text = "\n".join([
        f"[{i+1}] [Score: {row['score']:.2f}] {str(row['passage_text'])[:250]}..."
        for i, (_, row) in enumerate(top5.iterrows())
    ])
    
    # 填充 prompt
    prompt = FEW_SHOT_PROMPT_TEMPLATE.format(
        query=query,
        documents=docs_text
    )
    
    print(f"\n{'='*80}")
    print("🤖 Teacher Model 输出:")
    print("="*80)
    
    # 调用 API
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            max_tokens=1000
        )
        
        output = response.choices[0].message.content
        
        # 直接打印完整输出
        print(output)
        
        # 保存结果
        results.append({
            'qid': qid,
            'query': query,
            'ground_truth_preference': ground_truth_preference,
            'ground_truth_ratio': ground_truth_ratio,
            'teacher_output': output
        })
        
    except Exception as e:
        print(f"✗ API 调用失败: {e}")
        results.append({
            'qid': qid,
            'query': query,
            'ground_truth_preference': ground_truth_preference,
            'ground_truth_ratio': ground_truth_ratio,
            'error': str(e)
        })
    
    print("\n" + "="*80)
    
    # 延迟
    time.sleep(1)

# ============ 保存完整输出 ============
df_results = pd.DataFrame(results)
df_results.to_csv('teacher_outputs_10_queries_full.csv', index=False, encoding='utf-8')

print("\n" + "="*80)
print("✓ 测试完成！")
print("="*80)
print(f"结果已保存到: teacher_outputs_10_queries_full.csv")
print(f"成功生成: {len([r for r in results if 'teacher_output' in r])}/{len(results)}")
print("\n现在你可以:")
print("1. 查看上面的输出，评估 Teacher 的推理质量")
print("2. 检查 Analysis 是否详细、合理")
print("3. 看看 Decision 是否与 Ground Truth 一致")
print("4. 如果质量好，再扩展到全部 6980 个查询")

加载数据...
测试 User Study 的前 10 个查询

查询 1/10 - QID: 104861
Query: cost of interior concrete flooring
Ground Truth: PRF-Benefit (ratio: 0.713)

Top-5 文档:
  [1] Score: 25.10
      Even though each project is unique, interior building finishes for commercial buildings typically fall in an. average cost per square foot range. $5-$...
  [2] Score: 24.07
      We pour a lot of 3000 psi. concrete for interior concrete floors and 4000 psi concrete for exterior concrete. The concrete cost per yard of concrete i...
  [3] Score: 23.25
      How Much Does an Interior Door Cost? Typical costs: Prices start at about $20-$100 for a no-frills, interior door slab (no frame or hardware) or $40-$...
  [4] Score: 23.16
      for commercial concrete floors the cost is anywhere from $ 50 $ 1 00 dollar per sq ft more if you re trying to figure what the cost for concrete floor...
  [5] Score: 23.06
      The cost of the flooring, which ranges from about $4.50 per square foot and up. The cost of labor which typica